# THROO NYC DEMAND PREDICTION — COMPLETE ML PIPELINE

This notebook explores the Throo NYC ride dataset, trains and evaluates demand-level classifiers, and saves a model bundle for the Flask application.

**Scope:** this is a supervised classification estimate for the supplied surge and available-driver conditions. It is not a time-series forecast and does not predict a future time window.

## 1. Imports and reproducibility

The notebook uses the same two model inputs as the web app: `surge` and `available_driver_count`. Cross-validation and scaling are performed inside sklearn estimators to avoid preprocessing leakage.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
FEATURE_COLS = ["surge", "available_driver_count"]
DEMAND_MAP = {"Low": 0, "Medium": 1, "High": 2}
DEMAND_NAMES = {value: key for key, value in DEMAND_MAP.items()}
print("Imports ready; fixed random state:", RANDOM_STATE)

Imports ready; fixed random state: 42


## 2. Locate and load the dataset

The CSV is expected beside this notebook. If the notebook is launched from a different working directory, the project working directory is also checked.

In [2]:
DATA_PATH = Path("throo_nyc_demand.csv")
if not DATA_PATH.exists():
    candidates = [Path.cwd() / "throo_nyc_demand.csv", Path.cwd().parent / "throo_nyc_demand.csv"]
    DATA_PATH = next((path for path in candidates if path.exists()), DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError("Could not find throo_nyc_demand.csv. Run the notebook from the project folder or update DATA_PATH.")

raw_df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(raw_df):,} rows from {DATA_PATH.resolve()}")
display(raw_df.head())
print("Shape:", raw_df.shape)

Loaded 15,000 rows from D:\uber project\throo_nyc_demand\throo_nyc_demand.csv


,date,pickup_time,pickup_location,drop_location,distance,duration,ride_type,driver_rating,rider_rating,surge,...,driver_id,driver_latitude,driver_longitude,driver_available,driver_status,request_latitude,request_longitude,pickup_wait_time,cancellation_flag,available_driver_count
0,2025-03-16,05:30:14,Downtown,Suburbs,31.66,98.0,Standard,4.7,4.7,1.0,...,D3982,40.79546,-74.04151,0,Busy,40.78502,-74.03952,38.5,0,13
1,2025-03-22,14:03:31,Downtown,Downtown,10.87,32.0,Standard,4.7,4.5,1.0,...,D6352,40.76902,-73.94095,0,Busy,40.77979,-73.94965,27.9,0,24
2,2025-03-16,08:16:28,Downtown,Downtown,18.20,55.0,Standard,4.6,4.9,1.0,...,D7170,40.75414,-73.99255,1,Available,40.76850,-74.00629,16.3,1,18
3,2025-03-17,19:37:14,Downtown,Suburbs,21.56,59.0,Standard,4.9,4.6,2.2,...,D6687,40.77960,-73.98989,1,Available,40.76076,-74.00841,20.4,0,2
4,2025-01-20,14:12:23,Suburbs,Downtown,12.87,30.0,Pool,4.6,4.9,1.0,...,D1080,40.93884,-73.86098,1,Available,40.93428,-73.89532,17.9,1,12


Shape: (15000, 23)


## 3. Inspect data quality and target balance

In [3]:
required_cols = FEATURE_COLS + ["demand"]
missing_cols = [column for column in required_cols if column not in raw_df.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")

display(raw_df[required_cols].describe(include="all"))
print("\nRaw demand labels:")
display(raw_df["demand"].value_counts(dropna=False).to_frame("rows"))
print("\nMissing values in model columns:")
display(raw_df[required_cols].isna().sum().to_frame("missing"))

,surge,available_driver_count,demand
count,15000.000000,15000.000000,15000
unique,NaN,NaN,3
top,NaN,NaN,Medium
freq,NaN,NaN,8197
mean,1.286033,21.385467,NaN
std,0.474549,15.050355,NaN
min,1.000000,1.000000,NaN
25%,1.000000,11.000000,NaN
50%,1.000000,18.000000,NaN
75%,1.600000,29.000000,NaN



Raw demand labels:


,rows
demand,
Medium,8197
Low,4128
High,2675



Missing values in model columns:


,missing
surge,0
available_driver_count,0
demand,0


## 4. Clean features and encode demand labels

Demand labels are normalized case-insensitively. Numeric labels `0`, `1`, and `2` are accepted too. Rows with missing, malformed, or unsupported labels/features are excluded; exact duplicate rows are removed.

In [5]:
def normalize_demand(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        label = value.strip()
        if not label:
            return np.nan
        if label.lower() in {name.lower() for name in DEMAND_MAP}:
            return DEMAND_MAP[next(name for name in DEMAND_MAP if name.lower() == label.lower())]
        try:
            code = int(float(label))
        except ValueError:
            return np.nan
    elif isinstance(value, (int, float, np.integer, np.floating)):
        code = int(value)
    else:
        return np.nan
    return code if code in DEMAND_NAMES else np.nan

df = raw_df[required_cols].copy()
for column in FEATURE_COLS:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["demand"] = df["demand"].map(normalize_demand)
df = df.replace([np.inf, -np.inf], np.nan).drop_duplicates().dropna(subset=required_cols)
df["demand"] = df["demand"].astype(int)

if df.empty:
    raise ValueError("No valid training rows remain after cleaning.")
if df["demand"].nunique() < 2:
    raise ValueError("At least two demand classes are required for model training.")

print(f"Clean rows: {len(df):,} / {len(raw_df):,}")
print("Encoded demand counts:")
display(df["demand"].map(DEMAND_NAMES).value_counts().reindex(DEMAND_MAP).to_frame("rows"))
display(df.head())

Clean rows: 133 / 15,000
Encoded demand counts:


,rows
demand,
Low,34
Medium,64
High,35


,surge,available_driver_count,demand
0,1.0,13,1
1,1.0,24,1
2,1.0,18,1
3,2.2,2,2
4,1.0,12,1


## 5. Prepare a stratified holdout set

The test split is held aside for the final report only. Candidate selection uses cross-validation on the training split.

In [7]:
X = df[FEATURE_COLS]
y = df["demand"]
class_counts = y.value_counts()
if class_counts.min() < 5:
    raise ValueError("Each demand class needs at least five rows for the configured 5-fold stratified validation.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print("Training classes:", y_train.value_counts().sort_index().to_dict())

Training rows: 106; test rows: 27
Training classes: {0: 27, 1: 51, 2: 28}


## 6. Compare candidate classifiers

Logistic Regression scales features within its pipeline. Random Forest uses the original feature units. Both candidates are evaluated using weighted F1 on identical stratified folds.

In [8]:
candidates = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

cv_results = {}
for name, estimator in candidates.items():
    scores = cross_val_score(estimator, X_train, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1)
    cv_results[name] = {"mean_f1_weighted": float(scores.mean()), "std_f1_weighted": float(scores.std())}
    print(f"{name}: CV weighted F1 = {scores.mean():.4f} (+/- {scores.std():.4f})")

best_name = max(cv_results, key=lambda name: cv_results[name]["mean_f1_weighted"])
print("Selected candidate:", best_name)

Logistic Regression: CV weighted F1 = 0.9810 (+/- 0.0233)
Random Forest: CV weighted F1 = 1.0000 (+/- 0.0000)
Selected candidate: Random Forest


## 7. Tune Random Forest when it wins, then fit the selected model

The search is limited to the training data and uses the same stratified folds. The selected estimator is then fitted on the complete training split.

In [9]:
best_params = None
if best_name == "Random Forest":
    search = GridSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {"max_depth": [3, 6, None], "min_samples_leaf": [5, 20]},
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    model = search.best_estimator_
    best_params = search.best_params_
else:
    model = candidates[best_name]

model.fit(X_train, y_train)
print("Final estimator:", model)
print("Best parameters:", best_params)

Final estimator: RandomForestClassifier(max_depth=3, min_samples_leaf=5, n_jobs=-1,
                       random_state=42)
Best parameters: {'max_depth': 3, 'min_samples_leaf': 5}


## 8. Final holdout evaluation

The test set is evaluated once after model selection. Read precision, recall, and F1 per class alongside the confusion matrix; a high score is not evidence that the model predicts future demand.

In [10]:
y_pred = model.predict(X_test)
test_f1_weighted = float(f1_score(y_test, y_pred, average="weighted"))
labels = [DEMAND_MAP[name] for name in DEMAND_MAP]
target_names = list(DEMAND_MAP)
print(f"Holdout weighted F1: {test_f1_weighted:.4f}\n")
print(classification_report(y_test, y_pred, labels=labels, target_names=target_names, digits=4, zero_division=0))
print("Confusion matrix (rows = actual, columns = predicted):")
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=labels),
    index=[f"Actual {name}" for name in target_names],
    columns=[f"Predicted {name}" for name in target_names],
))

Holdout weighted F1: 1.0000

              precision    recall  f1-score   support

         Low     1.0000    1.0000    1.0000         7
      Medium     1.0000    1.0000    1.0000        13
        High     1.0000    1.0000    1.0000         7

    accuracy                         1.0000        27
   macro avg     1.0000    1.0000    1.0000        27
weighted avg     1.0000    1.0000    1.0000        27

Confusion matrix (rows = actual, columns = predicted):


,Predicted Low,Predicted Medium,Predicted High
Actual Low,7,0,0
Actual Medium,0,13,0
Actual High,0,0,7


## 9. Try a demand estimate

This applies the trained classifier to one hypothetical set of current conditions.

In [11]:
example = pd.DataFrame([{"surge": 1.2, "available_driver_count": 35}], columns=FEATURE_COLS)
class_probabilities = model.predict_proba(example)[0]
probabilities = {
    DEMAND_NAMES[int(class_id)]: round(float(probability), 4)
    for class_id, probability in zip(model.classes_, class_probabilities)
}
prediction = max(probabilities, key=probabilities.get)
print({"demand": prediction, "probabilities": probabilities})

{'demand': 'Low', 'probabilities': {'Low': 0.52, 'Medium': 0.48, 'High': 0.0}}


## 10. Save model artifacts for the web app

The bundle stores the fitted estimator as `model` with preprocessing included where needed, `needs_scaling=False`, and the feature list expected by `DemandPredictor`. The current Flask app will load it from `artifacts/demand_model.joblib`.

In [12]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / "demand_model.joblib"
metadata_path = ARTIFACT_DIR / "metadata.json"

joblib.dump({
    "model": model,
    "scaler": None,
    "needs_scaling": False,
    "feature_cols": FEATURE_COLS,
}, model_path)
metadata = {
    "best_model": best_name,
    "best_params": best_params,
    "cv_f1_weighted": {name: result["mean_f1_weighted"] for name, result in cv_results.items()},
    "test_f1_weighted": test_f1_weighted,
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "features": FEATURE_COLS,
    "classification_report": classification_report(
        y_test, y_pred, labels=labels, target_names=target_names, digits=4, zero_division=0
    ),
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Saved model: {model_path.resolve()}")
print(f"Saved metadata: {metadata_path.resolve()}")

Saved model: D:\uber project\throo_nyc_demand\artifacts\demand_model.joblib
Saved metadata: D:\uber project\throo_nyc_demand\artifacts\metadata.json


## 11. Load and smoke-test the saved bundle

Reload the artifact and check that it returns class probabilities in the format consumed by the app.

In [14]:
bundle = joblib.load(model_path)
loaded_model = bundle["model"]
loaded_probabilities = loaded_model.predict_proba(example)[0]
loaded_prediction = {
    DEMAND_NAMES[int(class_id)]: round(float(probability), 4)
    for class_id, probability in zip(loaded_model.classes_, loaded_probabilities)
}
assert set(bundle["feature_cols"]) == set(FEATURE_COLS)
assert abs(sum(loaded_prediction.values()) - 1.0) < 0.001
print({"demand": max(loaded_prediction, key=loaded_prediction.get), "probabilities": loaded_prediction})
print("Artifact smoke test passed.")

{'demand': 'Low', 'probabilities': {'Low': 0.52, 'Medium': 0.48, 'High': 0.0}}
Artifact smoke test passed.
